# 04 - Model Training Results

This notebook interprets existing training runs. It does not launch `BayesSearchCV` or score new stars. The primary decision view uses the fraction of holdout WR recovered in the top 100 candidates.


## Concepts Used

- `holdout`: final test set split by stable `source_id` hash before negative reduction.
- `CV`: stratified cross-validation over the training split.
- `BayesSearchCV`: uses CV for hyperparameter search; this is not redundant with reporting CV because the selected pipeline is later measured with out-of-fold predictions.
- `SMOTE` and `SMOTE-ENN`: applied after the holdout split and inside each training fold because they live inside an `imblearn.pipeline.Pipeline`.


In [ ]:
from pathlib import Path
import sys
import json

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Force an interactive notebook backend when running cells manually.
try:
    get_ipython().run_line_magic('matplotlib', 'inline')
except NameError:
    pass
from IPython.display import Markdown, Image, display

try:
    import seaborn as sns
except ImportError:
    sns = None

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.22,
    "font.size": 10,
})

PALETTE = {
    "xgboost": "#2F6F9F",
    "random_forest": "#3A9D5D",
    "hist_gradient_boosting": "#B7791F",
    "logistic_regression": "#7B61A8",
    "wr": "#A8324A",
    "candidate": "#2F6F9F",
    "negative": "#6B7280",
    "muted": "#6B7280",
}

def require_path(path):
    path = Path(path)
    if not path.is_absolute():
        path = ROOT / path
    if not path.exists():
        raise FileNotFoundError(path)
    return path

def pct(value):
    return "-" if pd.isna(value) else f"{100 * float(value):.1f}%"

def model_display(row):
    variant = str(row.get("dataset_variant", ""))
    family = "Relaxed" if variant.startswith("relaxed") else "Strict"
    dataset = variant.replace("relaxed_", "").replace("strict_", "")
    dataset = dataset.replace("photometry", "Phot").replace("parallax_soft", "Parallax+").replace("poe_", "POE>=")
    feat = "+err" if str(row.get("feature_set", "")).endswith("error") else "base"
    model = {"xgboost": "XGB", "random_forest": "RF", "hist_gradient_boosting": "HGB", "logistic_regression": "LogReg"}.get(str(row.get("model", "")), str(row.get("model", "")))
    sampler = {"smote": "SMOTE", "smote_enn": "SMOTE-ENN", "none": "no sampler"}.get(str(row.get("sampler", "")), str(row.get("sampler", "")))
    return f"{model} {sampler} | {family} {dataset} | {feat}"

def add_model_columns(df):
    out = df.copy()
    out["model_label"] = out.apply(model_display, axis=1)
    for k in [50, 100, 500, 1000]:
        c = f"holdout_wr_at_{k}"
        if c in out.columns:
            out[f"{c}_pct"] = out[c] / out["wr_holdout"].replace(0, np.nan)
    return out

def rank_models(df, k=100):
    return df.sort_values(
        [f"holdout_wr_at_{k}_pct", "holdout_average_precision", "holdout_recall_at_100", "holdout_f2_wr"],
        ascending=[False, False, False, False],
    ).reset_index(drop=True)

results = add_model_columns(results)
ranked = rank_models(results, k=100)
print(f"Configuraciones evaluadas: {len(results)}")
print(f"Aceptadas: {int(results['selection_status'].eq('accepted').sum())}")
print("Holdout WR by variant:")
display(results.drop_duplicates("dataset_variant").set_index("dataset_variant")["wr_holdout"].sort_index())


## Run Selection

This section lists runs synchronized in `training_history.duckdb`. Set `TRAINING_RUN_ID` to one available ID to inspect a specific training run.

In [ ]:
from wr_detector.modeling.notebook_runs import available_training_runs, load_training_results_for_notebook

available_runs = available_training_runs(ROOT)
display(available_runs)

TRAINING_RUN_ID = "run_202606_science_v2"
results = load_training_results_for_notebook(ROOT, TRAINING_RUN_ID)
display(Markdown(f"**Active run:** `{TRAINING_RUN_ID}` with `{len(results)}` results."))

In [ ]:

# Enrich current results with dynamic top-k recovery and source-level diagnostics.
HISTORY_DB = ROOT / "data/databases/training_history.duckdb"
WR_DB = ROOT / "data/databases/wr_reference.duckdb"


def wr_family(value):
    text = "" if pd.isna(value) else str(value).upper().strip()
    if not text:
        return "unknown"
    if "WN" in text and "WC" in text:
        return "WN/WC"
    if "WO" in text:
        return "WO"
    if "WC" in text:
        return "WC"
    if "WN" in text:
        return "WN"
    return "other/unknown"


def load_wr_catalog():
    if not WR_DB.exists():
        return pd.DataFrame(columns=["source_id", "wr_id", "spectral_type", "wr_family"])
    con = duckdb.connect(str(WR_DB), read_only=True)
    try:
        wr = con.execute('SELECT source_id, wr_id, "Spectral Type" AS spectral_type FROM wr_reference').fetchdf()
    finally:
        con.close()
    wr["wr_family"] = wr["spectral_type"].map(wr_family)
    return wr


WR_CATALOG = load_wr_catalog()


def load_predictions_for_row(row):
    if HISTORY_DB.exists():
        con = duckdb.connect(str(HISTORY_DB), read_only=True)
        try:
            cols = {r[1] for r in con.execute("PRAGMA table_info('model_predictions')").fetchall()}
            if {"source_id", "row_id"}.issubset(cols):
                pred = con.execute('''
                    SELECT split, row_id, source_id, target, score, predicted, threshold
                    FROM model_predictions
                    WHERE run_id = ? AND dataset_variant = ? AND feature_set = ? AND model = ? AND sampler = ?
                ''', [TRAINING_RUN_ID, row.dataset_variant, row.feature_set, row.model, row.sampler]).fetchdf()
                if not pred.empty:
                    return pred
        finally:
            con.close()
    pred_path = ROOT / str(row.get("predictions_path", ""))
    if pred_path.exists():
        return pd.read_csv(pred_path)
    return pd.DataFrame()


def load_holdout_with_predictions(row):
    pred = load_predictions_for_row(row)
    if pred.empty or "source_id" not in pred.columns:
        return pd.DataFrame()
    hold_pred = pred[pred["split"].eq("holdout")].copy()
    data_path = ROOT / "data/processed/modeling" / f"{row.dataset_variant}_reduced.parquet"
    if not data_path.exists():
        return pd.DataFrame()
    data = pd.read_parquet(data_path)
    hold = data[data["modeling_split"].eq("holdout")].copy()
    joined = hold.merge(hold_pred[["source_id", "score", "predicted", "threshold"]], on="source_id", how="inner")
    if not WR_CATALOG.empty:
        joined = joined.merge(WR_CATALOG, on="source_id", how="left", suffixes=("", "_catalog"))
        if "wr_id_catalog" in joined.columns:
            joined["wr_id"] = joined["wr_id"].fillna(joined["wr_id_catalog"])
    joined["spectral_type"] = joined.get("spectral_type", pd.Series("unknown", index=joined.index)).fillna("unknown")
    joined["wr_family"] = joined.get("wr_family", pd.Series("unknown", index=joined.index)).fillna("unknown")
    joined["rank"] = joined["score"].rank(method="first", ascending=False).astype(int)
    return joined


def add_dynamic_topk(results_df, ks=(10, 50, 100)):
    out = results_df.copy()
    for k in ks:
        out[f"holdout_wr_at_{k}_dynamic"] = np.nan
        out[f"holdout_wr_at_{k}_pct_dynamic"] = np.nan
    for idx, row in out.iterrows():
        joined = load_holdout_with_predictions(row)
        if joined.empty:
            continue
        ordered = joined.sort_values("score", ascending=False)
        wr_total = max(int((joined["target"] == 1).sum()), 1)
        for k in ks:
            recovered = int((ordered.head(k)["target"] == 1).sum())
            out.loc[idx, f"holdout_wr_at_{k}_dynamic"] = recovered
            out.loc[idx, f"holdout_wr_at_{k}_pct_dynamic"] = recovered / wr_total
    for k in ks:
        base_count = f"holdout_wr_at_{k}"
        base_pct = f"holdout_wr_at_{k}_pct"
        dyn_count = f"holdout_wr_at_{k}_dynamic"
        dyn_pct = f"holdout_wr_at_{k}_pct_dynamic"
        if base_count not in out.columns:
            out[base_count] = out[dyn_count]
        else:
            out[base_count] = out[base_count].fillna(out[dyn_count])
        if base_pct not in out.columns:
            out[base_pct] = out[dyn_pct]
        else:
            out[base_pct] = out[base_pct].fillna(out[dyn_pct])
    return out


def parse_best_params(value):
    if isinstance(value, dict):
        params = value
    elif pd.isna(value) or value == "":
        params = {}
    else:
        try:
            params = json.loads(value)
        except (TypeError, json.JSONDecodeError):
            params = {}
    return {str(k).replace("estimator__", ""): v for k, v in params.items()}


IMPORTANT_HYPERPARAMS = [
    "n_estimators", "max_depth", "min_samples_leaf", "max_features",
    "learning_rate", "subsample", "colsample_bytree", "reg_lambda", "min_child_weight",
]


def compact_hyperparams(row):
    pieces = []
    if pd.notna(row.get("n_estimators", np.nan)):
        pieces.append(f"trees={int(row['n_estimators'])}")
    if pd.notna(row.get("max_depth", np.nan)):
        pieces.append(f"depth={row['max_depth']}")
    if pd.notna(row.get("learning_rate", np.nan)):
        pieces.append(f"eta={float(row['learning_rate']):.3f}")
    if pd.notna(row.get("min_samples_leaf", np.nan)):
        pieces.append(f"leaf={row['min_samples_leaf']}")
    if pd.notna(row.get("min_child_weight", np.nan)):
        pieces.append(f"child={row['min_child_weight']}")
    if pd.notna(row.get("subsample", np.nan)):
        pieces.append(f"sub={float(row['subsample']):.2f}")
    if pd.notna(row.get("colsample_bytree", np.nan)):
        pieces.append(f"col={float(row['colsample_bytree']):.2f}")
    if pd.notna(row.get("reg_lambda", np.nan)):
        pieces.append(f"lambda={float(row['reg_lambda']):.2f}")
    return " | ".join(pieces) if pieces else "sin params"


def add_hyperparameter_columns(results_df):
    out = results_df.copy()
    parsed = out.get("bayes_best_params", pd.Series([{}] * len(out), index=out.index)).map(parse_best_params)
    numeric_params = [name for name in IMPORTANT_HYPERPARAMS if name != "max_features"]
    for name in IMPORTANT_HYPERPARAMS:
        out[name] = parsed.map(lambda params, n=name: params.get(n, np.nan))
        if name in numeric_params:
            out[name] = pd.to_numeric(out[name], errors="coerce")
    out["hyperparams_compact"] = out.apply(compact_hyperparams, axis=1)
    out["overfit_warning_flag"] = out["selection_status"].eq("overfit_warning")
    return out


results = add_hyperparameter_columns(results)
results = add_dynamic_topk(results, ks=(10, 50, 100))
ranked = rank_models(results, k=100)


## Primary Ranking: WR Recovered In The Top 100


In [ ]:
cols = [
    "model_label", "n_train", "wr_train", "n_holdout", "wr_holdout",
    "holdout_wr_at_100", "holdout_wr_at_100_pct", "holdout_average_precision",
    "holdout_f2_wr", "holdout_precision_wr", "holdout_recall_wr", "selection_status",
]
display(ranked[cols].head(15).style.format({
    "holdout_wr_at_100_pct": "{:.1%}",
    "holdout_average_precision": "{:.3f}",
    "holdout_f2_wr": "{:.3f}",
    "holdout_precision_wr": "{:.3f}",
    "holdout_recall_wr": "{:.3f}",
}))

fig, ax = plt.subplots(figsize=(11.5, 7))
plot = ranked.head(15).iloc[::-1]
colors = [PALETTE.get(m, PALETTE["muted"]) for m in plot["model"]]
ax.barh(plot["model_label"], 100 * plot["holdout_wr_at_100_pct"], color=colors)
for i, row in enumerate(plot.itertuples()):
    ax.text(100 * row.holdout_wr_at_100_pct + 1, i, f"{int(row.holdout_wr_at_100)}/{int(row.wr_holdout)}", va="center", fontsize=9)
ax.set_xlabel("WR recovered in top 100 / holdout WR (%)")
ax.set_title("Best configurations by top-100 WR recovery")
plt.tight_layout()
plt.show()


## Hyperparameters And Tree Complexity

Hyperparameters come from `BayesSearchCV`. This section does not change IDs or artifacts; it only extracts parameters to audit model capacity. Tree count helps interpret capacity, while overfit risk should still be read from the `train - CV` gap and `holdout - CV` stability.

In [ ]:

param_cols = [
    "model_label", "n_train", "wr_train", "n_holdout", "wr_holdout",
    "holdout_wr_at_10_pct", "holdout_wr_at_50_pct", "holdout_wr_at_100_pct",
    "holdout_average_precision", "cv_train_gap_f2", "holdout_cv_gap_f2", "selection_status",
    "hyperparams_compact",
]
display(ranked[param_cols].head(15).style.format({
    "holdout_wr_at_10_pct": "{:.1%}",
    "holdout_wr_at_50_pct": "{:.1%}",
    "holdout_wr_at_100_pct": "{:.1%}",
    "holdout_average_precision": "{:.3f}",
    "cv_train_gap_f2": "{:.3f}",
    "holdout_cv_gap_f2": "{:.3f}",
}))

tree_stats = (
    results.groupby("model", dropna=False)
    .agg(
        configs=("model_label", "size"),
        n_estimators_mean=("n_estimators", "mean"),
        n_estimators_median=("n_estimators", "median"),
        n_estimators_min=("n_estimators", "min"),
        n_estimators_max=("n_estimators", "max"),
        max_depth_mean=("max_depth", "mean"),
        max_depth_median=("max_depth", "median"),
        cv_train_gap_f2_mean=("cv_train_gap_f2", "mean"),
        cv_train_gap_f2_median=("cv_train_gap_f2", "median"),
        holdout_cv_gap_f2_mean=("holdout_cv_gap_f2", "mean"),
        holdout_ap_mean=("holdout_average_precision", "mean"),
        overfit_warning_rate=("overfit_warning_flag", "mean"),
    )
    .sort_values("holdout_ap_mean", ascending=False)
)
display(tree_stats.style.format({
    "n_estimators_mean": "{:.1f}",
    "n_estimators_median": "{:.0f}",
    "n_estimators_min": "{:.0f}",
    "n_estimators_max": "{:.0f}",
    "max_depth_mean": "{:.1f}",
    "max_depth_median": "{:.0f}",
    "cv_train_gap_f2_mean": "{:.3f}",
    "cv_train_gap_f2_median": "{:.3f}",
    "holdout_cv_gap_f2_mean": "{:.3f}",
    "holdout_ap_mean": "{:.3f}",
    "overfit_warning_rate": "{:.1%}",
}))

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.4))
for model_name, group in results.dropna(subset=["n_estimators"]).groupby("model"):
    color = PALETTE.get(model_name, PALETTE["muted"])
    axes[0].scatter(group["n_estimators"], group["cv_train_gap_f2"], s=52, alpha=0.78, color=color, label=model_name)
    axes[1].scatter(group["max_depth"], group["cv_train_gap_f2"], s=52, alpha=0.78, color=color, label=model_name)
for ax in axes:
    ax.axhline(0, color="#111827", lw=1, alpha=0.55)
    ax.axhline(0.15, color="#A8324A", lw=1, ls="--", alpha=0.65)
    ax.set_ylabel("gap F2 train - CV")
axes[0].set_xlabel("number of trees")
axes[1].set_xlabel("maximum depth")
axes[0].set_title("Trees versus overfit gap")
axes[1].set_title("Depth versus overfit gap")
axes[0].legend(frameon=False)
plt.tight_layout()
plt.show()


## Current Top Model


In [ ]:
for idx, row in ranked.head(3).reset_index(drop=True).iterrows():
    display(Markdown(
        f"**#{idx+1}: {row['model_label']}**  \n"
        f"- top 100: `{int(row['holdout_wr_at_100'])}/{int(row['wr_holdout'])}` = `{pct(row['holdout_wr_at_100_pct'])}`  \n"
        f"- train: `{int(row['n_train'])}` filas, `{int(row['wr_train'])}` WR  \n"
        f"- holdout/test: `{int(row['n_holdout'])}` filas, `{int(row['wr_holdout'])}` WR  \n"
        f"- AP holdout: `{row['holdout_average_precision']:.3f}`; F2 holdout: `{row['holdout_f2_wr']:.3f}`; estado: `{row['selection_status']}`"
    ))


## Ranking By Review Budget


In [ ]:
ks = [10, 50, 100, 500, 1000]
top = ranked.head(6)
fig, ax = plt.subplots(figsize=(10.5, 5.5))
for _, row in top.iterrows():
    ax.plot(ks, [100 * row.get(f"holdout_wr_at_{k}_pct", np.nan) for k in ks], marker="o", lw=2, label=row["model_label"])
ax.set_xscale("log")
ax.set_xticks(ks, labels=[str(k) for k in ks])
ax.set_ylabel("WR holdout recuperadas (%)")
ax.set_xlabel("candidates revisados")
ax.set_title("Cumulative recovery by review budget")
ax.legend(frameon=False, fontsize=8, loc="lower right")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(7.2, 5.4))
for model, group in results.groupby("model"):
    ax.scatter(100 * group["holdout_wr_at_100_pct"], group["holdout_average_precision"], s=55, alpha=0.75, label=model, color=PALETTE.get(model, PALETTE["muted"]))
ax.set_xlabel("WR top 100 / WR holdout (%)")
ax.set_ylabel("Average precision holdout")
ax.set_title("Ranking temprano vs precision promedio")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()


## Performance By Dataset Family


In [ ]:
variant_summary = ranked.groupby("dataset_variant").agg(
    best_top100_pct=("holdout_wr_at_100_pct", "max"),
    best_ap=("holdout_average_precision", "max"),
    wr_holdout=("wr_holdout", "first"),
    n_holdout=("n_holdout", "first"),
    configs=("model", "size"),
).sort_values("best_top100_pct", ascending=False)
display(variant_summary.style.format({"best_top100_pct": "{:.1%}", "best_ap": "{:.3f}"}))

heat = results.pivot_table(index="dataset_variant", columns=["model", "sampler"], values="holdout_wr_at_100_pct", aggfunc="max")
fig, ax = plt.subplots(figsize=(11.5, 5.5))
if sns:
    sns.heatmap(100 * heat, annot=True, fmt=".1f", cmap="YlGnBu", ax=ax, cbar_kws={"label": "% WR top 100"})
else:
    im = ax.imshow((100 * heat).fillna(0), cmap="YlGnBu")
    ax.set_xticks(range(len(heat.columns)), [" / ".join(c) for c in heat.columns], rotation=35, ha="right")
    ax.set_yticks(range(len(heat.index)), heat.index)
    plt.colorbar(im, ax=ax, label="% WR top 100")
ax.set_title("Best top-100 recovery by variant, model, and sampler")
plt.tight_layout()
plt.show()


## Rapid Overfit Diagnostic


In [ ]:
metrics = ["average_precision", "f2_wr", "recall_wr", "precision_wr"]
labels = ["Average precision", "F2", "Recall", "Precision"]
top = ranked.head(8)
fig, axes = plt.subplots(2, 2, figsize=(13, 8), sharex=True)
for ax, metric, label in zip(axes.ravel(), metrics, labels):
    x = np.arange(len(top))
    ax.plot(x, top[f"train_{metric}"], marker="o", label="train", color="#3A9D5D")
    ax.plot(x, top[f"cv_{metric}"], marker="o", label="CV", color="#B7791F")
    ax.plot(x, top[f"holdout_{metric}"], marker="o", label="holdout", color="#2F6F9F")
    ax.set_title(label)
    ax.set_xticks(x, top["model_label"], rotation=35, ha="right", fontsize=8)
axes[0, 0].legend(frameon=False)
fig.suptitle("Train vs CV vs holdout for the operational top model", y=1.02)
plt.tight_layout()
plt.show()


## SIMBAD Types Confused As WR

This section inspects holdout false positives: SIMBAD non-WR sources marked as WR at the selected threshold. They are grouped by `simbad_main_type` and `simbad_query_type` to characterize contaminant classes.

In [ ]:

rows = []
for _, row in ranked.head(10).iterrows():
    joined = load_holdout_with_predictions(row)
    if joined.empty:
        continue
    fp = joined[(joined["target"].eq(0)) & (joined["predicted"].eq(1))].copy()
    for col in ["simbad_main_type", "simbad_query_type"]:
        if col not in fp.columns:
            continue
        counts = fp[col].fillna("missing").value_counts().head(12)
        for label, count in counts.items():
            rows.append({
                "model_label": row["model_label"],
                "dataset_variant": row["dataset_variant"],
                "model": row["model"],
                "sampler": row["sampler"],
                "field": col,
                "type": label,
                "false_positive_rows": int(count),
                "total_false_positives": int(len(fp)),
                "share_of_fp": int(count) / max(len(fp), 1),
            })

fp_types = pd.DataFrame(rows)
if fp_types.empty:
    display(Markdown("No traceable false positives for the selected models."))
else:
    display(fp_types.style.format({"share_of_fp": "{:.1%}"}))
    plot = fp_types[fp_types["field"].eq("simbad_main_type")].head(25).iloc[::-1]
    fig, ax = plt.subplots(figsize=(11, 6))
    ax.barh(plot["model_label"] + " | " + plot["type"].astype(str), plot["false_positive_rows"], color="#A84545")
    ax.set_xlabel("holdout false positives")
    ax.set_title("SIMBAD types most often confused as WR")
    plt.tight_layout()
    plt.show()


## Least Recovered WR Subtypes

The subtype is taken from the Crowther/GWRC catalogue (`Spectral Type` in `wr_reference.duckdb`) joined by `source_id`. Recovery is reported by broad WR family and exact holdout spectral subtype.

In [ ]:

subtype_rows = []
family_rows = []
for _, row in ranked.head(10).iterrows():
    joined = load_holdout_with_predictions(row)
    if joined.empty:
        continue
    wr_holdout = joined[joined["target"].eq(1)].copy()
    if wr_holdout.empty:
        continue
    for group_col, collector in [("wr_family", family_rows), ("spectral_type", subtype_rows)]:
        grouped = wr_holdout.groupby(group_col, dropna=False).agg(
            wr_holdout=("source_id", "size"),
            wr_recovered=("predicted", "sum"),
            median_score=("score", "median"),
        ).reset_index().rename(columns={group_col: "wr_type"})
        grouped["recall_at_threshold"] = grouped["wr_recovered"] / grouped["wr_holdout"].replace(0, np.nan)
        grouped.insert(0, "model_label", row["model_label"])
        grouped.insert(1, "dataset_variant", row["dataset_variant"])
        collector.extend(grouped.to_dict("records"))

family_recovery = pd.DataFrame(family_rows)
subtype_recovery = pd.DataFrame(subtype_rows)
if family_recovery.empty:
    display(Markdown("No holdout WR rows can be traced to spectral subtype."))
else:
    display(Markdown("**Least recovered WR families by top model:**"))
    display(
        family_recovery.sort_values(["recall_at_threshold", "wr_holdout", "median_score"], ascending=[True, False, True])
        .head(30)
        .style.format({"recall_at_threshold": "{:.1%}", "median_score": "{:.3f}"})
    )
    display(Markdown("**Least recovered exact spectral subtypes (minimum 2 objects in model holdout):**"))
    subtype_view = subtype_recovery[subtype_recovery["wr_holdout"].ge(2)].sort_values(
        ["recall_at_threshold", "wr_holdout", "median_score"], ascending=[True, False, True]
    ).head(30)
    display(subtype_view.style.format({"recall_at_threshold": "{:.1%}", "median_score": "{:.3f}"}))

    top_model = ranked.iloc[0]
    top_family = family_recovery[family_recovery["model_label"].eq(top_model["model_label"])]
    if not top_family.empty:
        plot = top_family.sort_values("recall_at_threshold")
        fig, ax = plt.subplots(figsize=(8, 4.5))
        ax.barh(plot["wr_type"].astype(str), 100 * plot["recall_at_threshold"], color="#2F6F9F")
        for i, r in enumerate(plot.itertuples(index=False)):
            ax.text(100 * r.recall_at_threshold + 1, i, f"{int(r.wr_recovered)}/{int(r.wr_holdout)}", va="center", fontsize=9)
        ax.set_xlabel("threshold recovery (%)")
        ax.set_title(f"Recovered WR families - {top_model['model_label']}")
        plt.tight_layout()
        plt.show()


## Artefactos del top 3


In [ ]:
for idx, row in ranked.head(3).reset_index(drop=True).iterrows():
    display(Markdown(f"### #{idx+1}: {row['model_label']}"))
    display(Markdown(f"Top 100: `{int(row['holdout_wr_at_100'])}/{int(row['wr_holdout'])}` = `{pct(row['holdout_wr_at_100_pct'])}` | AP `{row['holdout_average_precision']:.3f}` | estado `{row['selection_status']}`"))
    for col in ["holdout_pr_curve_path", "holdout_roc_curve_path", "holdout_confusion_matrix_path", "feature_importance_figure_path"]:
        p = ROOT / str(row.get(col, ""))
        if p.exists():
            display(Image(filename=str(p)))
        else:
            display(Markdown(f"Missing `{col}`: `{p}`"))
